# 01 Data Audit and Manifest Preparation

This notebook audits the downloaded public datasets and identifies whether each dataset contains images, masks, annotations, metadata, or only classification-style folders.

The goal is to determine which datasets are actually usable for optic disc/cup segmentation.

In [7]:
from pathlib import Path
import sys
import os

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

src_path = PROJECT_ROOT / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

PROJECT_ROOT

PosixPath('/sfs/gpfs/tardis/home/gsr3qz/Documents/MSDS/CAPSTONE/GitHub/Automated-Glaucoma-Screening-Using-AI-Enhanced-Ophthalmoscopy')

In [8]:
DATASET_KEYS = [
    "glaucoma_fundus_imaging_bundle",
    "papila",
]

DATASET_KEYS

['glaucoma_fundus_imaging_bundle', 'papila']

In [9]:
from glaucoma_segmentation.data.discover_files import inventory_registry_datasets

inventory_df, summary_df, candidate_df = inventory_registry_datasets(
    dataset_keys=DATASET_KEYS,
    save_reports=True,
)

summary_df

,dataset_key,extension,file_class,file_count,total_size_bytes
2,glaucoma_fundus_imaging_bundle,.jpg,image,10051,5013137525
6,glaucoma_fundus_imaging_bundle,.png,possible_mask_image,8631,39319157
0,glaucoma_fundus_imaging_bundle,.bmp,possible_mask_image,1200,3886650400
3,glaucoma_fundus_imaging_bundle,.json,possible_annotation,1023,1977090
4,glaucoma_fundus_imaging_bundle,.mat,possible_annotation,650,16796084
1,glaucoma_fundus_imaging_bundle,.csv,possible_annotation,3,115247
8,glaucoma_fundus_imaging_bundle,.tif,possible_mask_image,2,50146
5,glaucoma_fundus_imaging_bundle,.pkl,other,1,814
7,glaucoma_fundus_imaging_bundle,.pth,other,1,1107541735
15,papila,.txt,possible_annotation,1963,1091265


In [10]:
candidate_df[
    [
        "dataset_key",
        "relative_path",
        "extension",
        "file_class",
        "has_mask_keyword",
        "file_size_bytes",
    ]
].head(100)

,dataset_key,relative_path,extension,file_class,has_mask_keyword,file_size_bytes
0,glaucoma_fundus_imaging_bundle,G1020/G1020.csv,.csv,possible_annotation,False,18036
2,glaucoma_fundus_imaging_bundle,G1020/Images/image_0.json,.json,possible_annotation,False,1083
4,glaucoma_fundus_imaging_bundle,G1020/Images/image_1.json,.json,possible_annotation,False,1157
6,glaucoma_fundus_imaging_bundle,G1020/Images/image_10.json,.json,possible_annotation,False,848
8,glaucoma_fundus_imaging_bundle,G1020/Images/image_1000.json,.json,possible_annotation,False,903
...,...,...,...,...,...,...
190,glaucoma_fundus_imaging_bundle,G1020/Images/image_1260.json,.json,possible_annotation,False,895
192,glaucoma_fundus_imaging_bundle,G1020/Images/image_1261.json,.json,possible_annotation,False,1416
194,glaucoma_fundus_imaging_bundle,G1020/Images/image_1270.json,.json,possible_annotation,False,1731
196,glaucoma_fundus_imaging_bundle,G1020/Images/image_1271.json,.json,possible_annotation,False,1062


In [11]:
summary_df.sort_values(
    ["dataset_key", "file_count"],
    ascending=[True, False],
)

,dataset_key,extension,file_class,file_count,total_size_bytes
2,glaucoma_fundus_imaging_bundle,.jpg,image,10051,5013137525
6,glaucoma_fundus_imaging_bundle,.png,possible_mask_image,8631,39319157
0,glaucoma_fundus_imaging_bundle,.bmp,possible_mask_image,1200,3886650400
3,glaucoma_fundus_imaging_bundle,.json,possible_annotation,1023,1977090
4,glaucoma_fundus_imaging_bundle,.mat,possible_annotation,650,16796084
1,glaucoma_fundus_imaging_bundle,.csv,possible_annotation,3,115247
8,glaucoma_fundus_imaging_bundle,.tif,possible_mask_image,2,50146
5,glaucoma_fundus_imaging_bundle,.pkl,other,1,814
7,glaucoma_fundus_imaging_bundle,.pth,other,1,1107541735
15,papila,.txt,possible_annotation,1963,1091265


In [12]:
dataset_summary = (
    inventory_df
    .groupby(["dataset_key", "file_class"])
    .size()
    .reset_index(name="file_count")
    .pivot(index="dataset_key", columns="file_class", values="file_count")
    .fillna(0)
    .astype(int)
)

dataset_summary

file_class,image,other,possible_annotation,possible_mask_image
dataset_key,,,,
glaucoma_fundus_imaging_bundle,10051,2,1676,9833
papila,488,4,1985,488


## Initial audit complete

The next step is to review the candidate masks/annotations and determine which datasets can support optic disc/cup segmentation.

After that, the manifest-building portion of this notebook will create structured image/mask pair records.